# cpp-lob-engine — microstructure analytics

End-to-end demo: drive the C++ matching engine from Python via `lobpy`, replay a
synthetic order-flow simulation, reconstruct the book + trade tape, and build
standard microstructure features (mid / micro-price, order-flow imbalance,
queue imbalance, Lee-Ready trade signs, realized spread, mark-outs).

This notebook is executed headlessly in CI-style via
`jupyter nbconvert --to notebook --execute`.

In [ ]:
import sys, pathlib
# Make the compiled lobpy module and the features helper importable.
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'analytics' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'build'))
sys.path.insert(0, str(ROOT / 'analytics'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lobpy
import features as feat
print('lobpy loaded from', lobpy.__file__)

## 1. Simulate and build features

In [ ]:
sim = lobpy.simulate(n=150_000, seed=11, p_aggressive=0.2, p_market=0.08)
f = feat.build_features(sim, markout_horizons=(1, 5, 20, 100))
l1, trades, mo = f['l1'], f['trades'], f['markouts']
print(f'L1 rows = {len(l1):,}   trades = {len(trades):,}')
l1.head()

## 2. Book / price evolution: mid vs micro-price and spread

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
w = slice(0, 4000)
ax1.plot(l1.mid.values[w], label='mid', lw=0.9)
ax1.plot(l1.micro_price.values[w], label='micro-price', lw=0.9, alpha=0.8)
ax1.set_ylabel('price (ticks)'); ax1.legend(); ax1.set_title('Mid vs micro-price')
ax2.plot(l1.spread.values[w], lw=0.6, color='#d62728')
ax2.set_ylabel('spread (ticks)'); ax2.set_xlabel('message index')
ax2.set_title('Quoted spread')
plt.tight_layout(); plt.show()

## 3. Depth imbalance over time\nQueue imbalance = (bid_size - ask_size)/(bid+ask); leads short-horizon price moves.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(l1.queue_imbalance.values[0:4000], lw=0.5, color='#2ca02c')
ax.axhline(0, color='grey', lw=0.7)
ax.set_ylabel('queue imbalance'); ax.set_xlabel('message index')
ax.set_title('Top-of-book queue imbalance'); plt.tight_layout(); plt.show()

## 4. Order-flow imbalance (OFI) vs short-horizon return\nClassic result: OFI is contemporaneously/▶-predictive of the next mid move.

In [ ]:
sub = l1.dropna(subset=['ofi', 'fwd_ret_10'])
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(sub.ofi, sub.fwd_ret_10, s=3, alpha=0.15, color='#1f77b4')
b, a = np.polyfit(sub.ofi, sub.fwd_ret_10, 1)
xs = np.linspace(sub.ofi.min(), sub.ofi.max(), 50)
corr = np.corrcoef(sub.ofi, sub.fwd_ret_10)[0, 1]
ax.plot(xs, a + b*xs, color='#d62728', label=f'slope={b:.2e}, corr={corr:.3f}')
ax.set_xlabel('OFI'); ax.set_ylabel('forward 10-msg mid return (ticks)')
ax.set_title('OFI vs forward return'); ax.legend(); plt.tight_layout(); plt.show()
print('OFI / forward-return correlation:', round(float(corr), 4))

## 5. Mark-out curve\nAverage sign-adjusted mid drift after a trade, by horizon — a proxy for adverse selection / information content.

In [ ]:
horizons = [int(c[1:]) for c in mo.columns]
avg = [mo[c].mean() for c in mo.columns]
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(horizons, avg, marker='o', color='#9467bd')
ax.axhline(0, color='grey', lw=0.7)
ax.set_xlabel('horizon (messages)'); ax.set_ylabel('avg signed mark-out (ticks)')
ax.set_title('Mark-out curve'); plt.tight_layout(); plt.show()
pd.DataFrame({'horizon': horizons, 'avg_markout_ticks': avg})

## 6. Trade-sign + realized spread summary

In [ ]:
buys = int((trades.sign > 0).sum()); sells = int((trades.sign < 0).sum())
print(f'Lee-Ready signed trades  buy={buys:,}  sell={sells:,}')
print(f'mean effective realized spread (h=20): {trades.realized_spread_h20.mean():.3f} ticks')
print(f'mean quoted spread: {l1.spread.mean():.3f} ticks')
trades[['ts','price','qty','sign','realized_spread_h20']].head()